# 03 - DnD DM World-State Engine (Verbose Trace)

This notebook packages the loop into an engine class with persistent multi-turn state.

Reading order:
1. Setup, adapter config, and trace toggles.
2. World model builders and serialization helpers.
3. Engine internals (tools, policies, loop, persistence).
4. Multi-turn demo run.

Design goal:
- move from one-off loop scripts to a reusable DM runtime object.


## Step 1 - Environment and Adapter Setup

Imports and global trace/model settings used by the engine.


In [ ]:
# Section: Setup
# Purpose: Import dependencies and define shared model/trace configuration for the engine.

import json
import random
import re
import sys
import time
from pathlib import Path
from typing import Any

# Ensure repo root is importable when running this notebook from its folder.
repo_root = Path.cwd()
while repo_root != repo_root.parent and not (repo_root / "orchestrator").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from orchestrator.llm_interaction.adapter import LLMAdapter, LLMError

MODEL = "gpt-oss:20b"
MAX_ITERATIONS = 60
TRACE_PREVIEW_CHARS = 500

# Maximum-detail tracing controls
FULL_TRACE = True
SHOW_THINKING_TRACE = True
SHOW_RAW_RESPONSE = True
SHOW_RESPONSE_STATS = True
PRINT_FULL_TRACE_AFTER_RUN = True
PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN = True

adapter = LLMAdapter(model=MODEL, verbose=False)


## Step 2 - World Model Utilities

Defines initial state shape and helper renderers used throughout the engine.


In [ ]:
# Section: World utilities
# Purpose: Build initial world state and utility serializers used by engine methods.

from copy import deepcopy


def build_initial_world_state() -> dict[str, Any]:
    return {
        "setting": "Ashen Coast",
        "location": {
            "name": "Port Ember",
            "description": "A storm-battered harbor city built on black volcanic stone.",
        },
        "party": {
            "name": "The Lantern Company",
            "members": ["Kara (Rogue)", "Brann (Cleric)", "Ilya (Wizard)"],
            "inventory": ["50 gp", "healing potion", "smoke bomb"],
        },
        "npcs": {},
        "quests": [],
        "log": [],
    }


def world_summary(state: dict[str, Any]) -> str:
    return json.dumps(
        {
            "setting": state["setting"],
            "location": state["location"],
            "party": state["party"],
            "npcs": state["npcs"],
            "quests": state["quests"],
        },
        indent=2,
    )


## Step 3 - Engine Implementation

`DungeonMasterEngine` includes:
- tool schemas + implementations
- tool dispatcher
- stop policy
- iterative run loop per turn
- persistent history and world updates


In [ ]:
# Section: Engine core
# Purpose: Implement persistent DM runtime with tools, policy checks, and verbose loop tracing.

RUN_STATE = {
    "total_tool_calls": 0,
}


def reset_run_state() -> None:
    RUN_STATE["total_tool_calls"] = 0


def ts() -> str:
    return time.strftime("%H:%M:%S")


def preview(text: str, limit: int = TRACE_PREVIEW_CHARS) -> str:
    if text is None:
        return ""
    text = str(text).strip()
    if FULL_TRACE:
        return text
    if len(text) <= limit:
        return text
    return text[:limit] + f" ... [truncated {len(text) - limit} chars]"


def trace_print(tag: str, message: str, trace_lines: list[str] | None = None) -> None:
    line = f"[{ts()}] [{tag}] {message}"
    print(line, flush=True)
    if trace_lines is not None:
        trace_lines.append(line)


def response_to_dict(response: Any) -> dict[str, Any]:
    if hasattr(response, "model_dump"):
        try:
            data = response.model_dump(exclude_none=False)
            if isinstance(data, dict):
                return data
        except Exception:
            pass

    if isinstance(response, dict):
        return response

    return {
        "_type": type(response).__name__,
        "repr": repr(response),
    }


def extract_message_dict(response: Any) -> dict[str, Any]:
    data = response_to_dict(response)
    msg = data.get("message") if isinstance(data, dict) else None
    if isinstance(msg, dict):
        return msg
    if hasattr(msg, "model_dump"):
        try:
            dumped = msg.model_dump(exclude_none=False)
            if isinstance(dumped, dict):
                return dumped
        except Exception:
            pass
    return {}


def extract_thinking_text(response: Any) -> str:
    msg = extract_message_dict(response)
    thinking = msg.get("thinking", "")

    if isinstance(thinking, list):
        return "".join(str(x) for x in thinking)

    return str(thinking or "")


def normalize_tool_calls(response: Any) -> list[dict[str, Any]]:
    out: list[dict[str, Any]] = []
    raw_calls = adapter._extract_tool_calls(response)

    for i, call in enumerate(raw_calls):
        if not isinstance(call, dict):
            continue

        fn = call.get("function", {})
        name = fn.get("name") or call.get("name")
        args = fn.get("arguments", {})

        if isinstance(args, str):
            try:
                args = json.loads(args)
            except json.JSONDecodeError:
                args = {}

        if not isinstance(args, dict):
            args = {}

        if not name:
            continue

        out.append(
            {
                "id": call.get("id") or f"call_{i}",
                "name": str(name),
                "arguments": args,
                "raw": call,
                "source": "structured",
            }
        )

    return out


def extract_fallback_tool_calls_from_text(text: str) -> list[dict[str, Any]]:
    if not text:
        return []

    calls: list[dict[str, Any]] = []

    for line in text.splitlines():
        stripped = line.strip().rstrip(",")
        if not stripped.startswith("{") or not stripped.endswith("}"):
            continue
        try:
            obj = json.loads(stripped)
        except json.JSONDecodeError:
            continue
        if not isinstance(obj, dict):
            continue
        name = obj.get("name")
        params = obj.get("parameters", {})
        if not name or not isinstance(params, dict):
            continue
        calls.append(
            {
                "id": f"text_line_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {
                    "function": {
                        "name": str(name),
                        "arguments": params,
                    }
                },
                "source": "text_fallback",
            }
        )

    pattern = re.compile(
        r'\{\s*"name"\s*:\s*"(?P<name>[^"]+)"\s*,\s*"parameters"\s*:\s*(?P<params>\{.*?\})\s*\}',
        flags=re.DOTALL,
    )

    for match in pattern.finditer(text):
        name = match.group("name")
        params_raw = match.group("params")
        try:
            params = json.loads(params_raw)
        except json.JSONDecodeError:
            continue
        if not isinstance(params, dict):
            continue

        duplicate = any(c["name"] == str(name) and c["arguments"] == params for c in calls)
        if duplicate:
            continue

        calls.append(
            {
                "id": f"text_regex_{len(calls)}",
                "name": str(name),
                "arguments": params,
                "raw": {
                    "function": {
                        "name": str(name),
                        "arguments": params,
                    }
                },
                "source": "text_fallback",
            }
        )

    return calls


class DungeonMasterEngine:
    def __init__(self, model: str = MODEL):
        self.model = model
        self.adapter = LLMAdapter(model=model)
        self.state = build_initial_world_state()
        self.system_prompt = (
            "You are a DnD dungeon master with strict state discipline. "
            "Use tools for any world mutation or state lookup. "
            "Never invent tool outputs. Emit detailed reasoning in your thinking stream. "
            "End each final response with 2-3 player choices."
        )
        self.history: list[dict[str, Any]] = [
            {
                "role": "user",
                "content": "Initial world state:\n" + world_summary(self.state),
            },
        ]
        self.last_turn_trace: list[str] = []
        self.last_response_snapshots: list[dict[str, Any]] = []
        self.total_tool_calls = 0

        self.tools = self._build_tools()
        self.tool_impl = {
            "read_world": self.read_world,
            "update_location": self.update_location,
            "create_npc": self.create_npc,
            "set_npc_attitude": self.set_npc_attitude,
            "add_quest": self.add_quest,
            "resolve_quest": self.resolve_quest,
            "roll_dice": self.roll_dice,
            "add_log_entry": self.add_log_entry,
        }

    def _build_tools(self) -> list[dict[str, Any]]:
        return [
            {
                "type": "function",
                "function": {
                    "name": "read_world",
                    "description": "Read the full world state or a specific top-level section.",
                    "parameters": {
                        "type": "object",
                        "properties": {"section": {"type": "string"}},
                        "required": [],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "update_location",
                    "description": "Set current location name and description.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "description": {"type": "string"},
                        },
                        "required": ["name", "description"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "create_npc",
                    "description": "Create or replace an NPC.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "role": {"type": "string"},
                            "attitude": {"type": "string"},
                        },
                        "required": ["name", "role", "attitude"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "set_npc_attitude",
                    "description": "Update an existing NPC attitude.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "name": {"type": "string"},
                            "attitude": {"type": "string"},
                        },
                        "required": ["name", "attitude"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "add_quest",
                    "description": "Add a quest with open status.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "objective": {"type": "string"},
                        },
                        "required": ["title", "objective"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "resolve_quest",
                    "description": "Mark quest as resolved with outcome text.",
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "title": {"type": "string"},
                            "outcome": {"type": "string"},
                        },
                        "required": ["title", "outcome"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "roll_dice",
                    "description": "Roll NdM(+/-K), example 2d6+1.",
                    "parameters": {
                        "type": "object",
                        "properties": {"expression": {"type": "string"}},
                        "required": ["expression"],
                    },
                },
            },
            {
                "type": "function",
                "function": {
                    "name": "add_log_entry",
                    "description": "Append a structured note to campaign log.",
                    "parameters": {
                        "type": "object",
                        "properties": {"text": {"type": "string"}},
                        "required": ["text"],
                    },
                },
            },
        ]

    # ---- tool implementations ----
    def read_world(self, section: str | None = None) -> dict[str, Any]:
        if not section:
            return deepcopy(self.state)
        if section not in self.state:
            raise ValueError(f"Unknown section '{section}'")
        return deepcopy(self.state[section])

    def update_location(self, name: str, description: str) -> dict[str, Any]:
        self.state["location"] = {"name": name, "description": description}
        return self.state["location"]

    def create_npc(self, name: str, role: str, attitude: str) -> dict[str, Any]:
        self.state["npcs"][name] = {"role": role, "attitude": attitude}
        return {"name": name, **self.state["npcs"][name]}

    def set_npc_attitude(self, name: str, attitude: str) -> dict[str, Any]:
        if name not in self.state["npcs"]:
            raise ValueError(f"NPC '{name}' does not exist")
        self.state["npcs"][name]["attitude"] = attitude
        return {"name": name, **self.state["npcs"][name]}

    def add_quest(self, title: str, objective: str) -> dict[str, Any]:
        quest = {"title": title, "objective": objective, "status": "open"}
        self.state["quests"].append(quest)
        return quest

    def resolve_quest(self, title: str, outcome: str) -> dict[str, Any]:
        for q in self.state["quests"]:
            if q["title"] == title:
                q["status"] = "resolved"
                q["outcome"] = outcome
                return q
        raise ValueError(f"Quest '{title}' not found")

    def roll_dice(self, expression: str) -> dict[str, Any]:
        m = re.fullmatch(r"(\d+)d(\d+)([+-]\d+)?", expression.strip())
        if not m:
            raise ValueError(f"Invalid dice expression: {expression}")
        n, sides, mod = int(m.group(1)), int(m.group(2)), int(m.group(3) or 0)
        rolls = [random.randint(1, sides) for _ in range(n)]
        return {
            "expression": expression,
            "rolls": rolls,
            "modifier": mod,
            "total": sum(rolls) + mod,
        }

    def add_log_entry(self, text: str) -> dict[str, Any]:
        entry = {"text": text}
        self.state["log"].append(entry)
        return entry

    def _execute_tool(self, name: str, arguments: dict[str, Any], trace_lines: list[str]) -> dict[str, Any]:
        self.total_tool_calls += 1
        trace_print("TOOL-CALL", f"#{self.total_tool_calls} {name} args={preview(json.dumps(arguments, ensure_ascii=True))}", trace_lines)

        fn = self.tool_impl.get(name)
        if not fn:
            payload = {"ok": False, "error": f"Unknown tool: {name}"}
            trace_print("TOOL-RESULT", f"#{self.total_tool_calls} {name} -> {payload['error']}", trace_lines)
            return payload

        try:
            payload = {"ok": True, "result": fn(**arguments)}
        except Exception as exc:
            payload = {"ok": False, "error": str(exc)}

        trace_print("TOOL-RESULT", f"#{self.total_tool_calls} {name} -> {preview(json.dumps(payload, ensure_ascii=True))}", trace_lines)
        return payload

    def _stop_hook(self, assistant_text: str, stop_hook_active: bool) -> str | None:
        text = (assistant_text or "").lower()
        has_choices = "what do you do" in text or "choices" in text or "options" in text
        if not has_choices:
            return "Provide 2-3 concrete player choices before completing the turn."

        if stop_hook_active and len(assistant_text.strip()) < 120:
            return "When stop hook is active, provide more detailed scene resolution."

        return None

    def run_turn(self, player_input: str, max_iterations: int = MAX_ITERATIONS) -> dict[str, Any]:
        trace_lines: list[str] = []
        response_snapshots: list[dict[str, Any]] = []
        stop_hook_active = False

        messages = [
            *self.history,
            {"role": "user", "content": f"Player action: {player_input}"},
        ]

        trace_print("RUN", f"model={self.model} max_iterations={max_iterations}", trace_lines)

        for iteration in range(1, max_iterations + 1):
            trace_print("ITER", f"{iteration}/{max_iterations} stop_hook_active={stop_hook_active}", trace_lines)

            try:
                response = self.adapter.request_with_tools(
                    stage="notebook_03_verbose",
                    system_prompt=self.system_prompt,
                    messages=messages,
                    tools=self.tools,
                )
            except LLMError as exc:
                trace_print("MODEL-ERROR", str(exc), trace_lines)
                return {
                    "status": "error",
                    "narration": str(exc),
                    "turn_trace": trace_lines,
                    "response_snapshots": response_snapshots,
                }

            response_dict = response_to_dict(response)
            response_snapshots.append(response_dict)

            if SHOW_RESPONSE_STATS:
                stats = {
                    "model": response_dict.get("model"),
                    "done_reason": response_dict.get("done_reason"),
                    "prompt_eval_count": response_dict.get("prompt_eval_count"),
                    "eval_count": response_dict.get("eval_count"),
                    "total_duration": response_dict.get("total_duration"),
                }
                trace_print("MODEL-STATS", json.dumps(stats, ensure_ascii=True), trace_lines)

            if SHOW_RAW_RESPONSE:
                trace_print("RAW-RESPONSE", json.dumps(response_dict, indent=2, ensure_ascii=True), trace_lines)

            assistant_text = self.adapter._extract_content(response).strip()
            thinking_text = extract_thinking_text(response).strip()

            trace_print("ASSISTANT-CONTENT", preview(assistant_text or "<empty>"), trace_lines)
            if SHOW_THINKING_TRACE:
                trace_print("ASSISTANT-THINKING", preview(thinking_text or "<none>"), trace_lines)

            tool_calls = normalize_tool_calls(response)
            if not tool_calls:
                fallback_source = "\n".join(x for x in [assistant_text, thinking_text] if x)
                fallback_calls = extract_fallback_tool_calls_from_text(fallback_source)
                if fallback_calls:
                    tool_calls = fallback_calls
                    trace_print("FALLBACK", f"Recovered {len(tool_calls)} text-encoded tool calls.", trace_lines)

            if tool_calls:
                messages.append(
                    {
                        "role": "assistant",
                        "content": assistant_text,
                        "tool_calls": [c["raw"] for c in tool_calls],
                    }
                )

                for call in tool_calls:
                    trace_print("TOOL-SOURCE", f"{call['name']} via {call.get('source', 'unknown')}", trace_lines)
                    payload = self._execute_tool(call["name"], call["arguments"], trace_lines)
                    messages.append(
                        {
                            "role": "tool",
                            "tool_name": call["name"],
                            "content": json.dumps(payload, separators=(",", ":"), ensure_ascii=True),
                        }
                    )

                trace_print("PROGRESS", f"total_tool_calls={self.total_tool_calls}", trace_lines)
                continue

            reason = self._stop_hook(assistant_text, stop_hook_active)
            if reason:
                stop_hook_active = True
                trace_print("HOOK-STOP", reason, trace_lines)
                messages.append({"role": "assistant", "content": assistant_text})
                messages.append(
                    {
                        "role": "user",
                        "content": f"You were about to finish but stop hook blocked completion: {reason}",
                    }
                )
                continue

            messages.append({"role": "assistant", "content": assistant_text})
            self.history = messages
            self.state["log"].append({"text": assistant_text, "type": "narration"})
            self.last_turn_trace = trace_lines
            self.last_response_snapshots = response_snapshots

            return {
                "status": "completed",
                "narration": assistant_text,
                "turn_trace": trace_lines,
                "response_snapshots": response_snapshots,
                "tool_calls_total": self.total_tool_calls,
            }

        self.history = messages
        self.last_turn_trace = trace_lines
        self.last_response_snapshots = response_snapshots
        return {
            "status": "max_iterations",
            "narration": "Stopped after max iterations.",
            "turn_trace": trace_lines,
            "response_snapshots": response_snapshots,
            "tool_calls_total": self.total_tool_calls,
        }


## Step 4 - Multi-Turn Demo

Runs multiple player turns through the same engine instance so you can observe state continuity.


In [ ]:
# Section: Demo execution
# Purpose: Run a multi-turn scenario and inspect evolving world state and response traces.

engine = DungeonMasterEngine()

turns = [
    "We ask around Port Ember for rumors about smugglers.",
    "I bribe the dock clerk and ask where the Midnight Lantern docks.",
    "We follow the lead to the warehouse district and prepare an ambush.",
]

all_snapshots: list[dict[str, Any]] = []

for i, prompt in enumerate(turns, start=1):
    out = engine.run_turn(prompt)
    print(f"\n===== TURN {i} ({out['status']}) =====")
    print(out["narration"])
    print("tool calls total:", out.get("tool_calls_total"))

    turn_trace = out.get("turn_trace", [])
    print("trace lines:", len(turn_trace))
    if PRINT_FULL_TRACE_AFTER_RUN:
        for line in turn_trace:
            print(line)

    turn_snaps = out.get("response_snapshots", [])
    all_snapshots.extend(turn_snaps)

print("\n===== WORLD STATE SNAPSHOT =====\n")
print(world_summary(engine.state))

print("\n===== RESPONSE SNAPSHOT COUNT =====")
print(len(all_snapshots))
if PRINT_FULL_RESPONSE_SNAPSHOTS_AFTER_RUN:
    for i, snap in enumerate(all_snapshots, start=1):
        print(f"\n--- SNAPSHOT {i} ---")
        print(json.dumps(snap, indent=2, ensure_ascii=True))
